# Fine-Tuning BERT for POS Tagging & Chunking

Here we'll do token classification: assigning a label to every token.

Tasks:
- **POS Tagging**: Noun, verb, adjective, etc.
- **Chunking**: Grouping into phrases (noun phrases, etc.).

We use **DistilBERT** because it's lighter and faster than BERT, perfect for a standard GPU.

## 1. Dataset Selection

We use **CoNLL-2003**, the standard for token classification.
- **POS tags**: `NN` (Noun), `VB` (Verb).
- **Chunk tags**: IOB format (e.g., `B-NP` for beginning of Noun Phrase).

Let's load the data.

In [ ]:
# Install required libraries if you haven't already
!pip install transformers datasets torch seqeval evaluate

from datasets import load_dataset

# Loading the CoNLL-2003 dataset
dataset = load_dataset('conll2003')

print("Dataset structure:\n", dataset)

# Let's check out the POS and Chunk features
pos_features = dataset['train'].features['pos_tags'].feature
chunk_features = dataset['train'].features['chunk_tags'].feature

print("\nNumber of POS tags:", pos_features.num_classes)
print("Number of Chunk tags:", chunk_features.num_classes)

# Looking at an actual example
example = dataset['train'][0]
print("\nFirst sentence tokens:", example['tokens'])
print("POS tag IDs:", example['pos_tags'])
print("Chunk tag IDs:", example['chunk_tags'])

## 2. Data Preprocessing

Transformers split rare words into subwords (e.g., `['transform', '##ers']`). 
Since our labels are per-word, we must align them with these subwords:
1. Keep the original label for the first subword.
2. Assign `-100` to the rest (and special tokens like `[CLS]`) so PyTorch's loss ignores them.

We'll focus on **Chunking** for this run.

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples, label_column='chunk_tags'):
    # Tokenize the input words. is_split_into_words=True because our data is already tokenized into word lists
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples[label_column]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Maps subwords to original word indices
        previous_word_idx = None
        label_ids = []
        
        for word_idx in word_ids:
            if word_idx is None:
                # Special tokens like [CLS] and [SEP]
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # First subword of a word gets the actual label
                label_ids.append(label[word_idx])
            else:
                # Subsequent subwords of the same word get -100
                label_ids.append(-100)
            previous_word_idx = word_idx
            
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Apply the preprocessing to the entire dataset
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

# Let's verify what the output looks like
sample = tokenized_datasets['train'][0]
print("Subword Tokens:", tokenizer.convert_ids_to_tokens(sample["input_ids"]))
print("Input IDs:", sample["input_ids"])
print("Attention Mask:", sample["attention_mask"])
print("Aligned Labels:", sample["labels"])

## 3. Model Setup

We use `AutoModelForTokenClassification` with DistilBERT. 
We also pass our `label2id` mapping so the model understands the output dimensions.

In [ ]:
from transformers import AutoModelForTokenClassification

# We'll extract the label list for chunking
label_list = dataset["train"].features["chunk_tags"].feature.names
num_labels = len(label_list)

# Creating mappings from ID to Label and vice-versa
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

print(f"Model initialized with {num_labels} classification labels.")

## 4. Training

Using the Hugging Face `Trainer` API makes this easy.
- **LR**: `2e-5`
- **Batch**: `16`
- **Epochs**: `3`
- **Weight Decay**: `0.01`

In [ ]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification

batch_size = 16

args = TrainingArguments(
    output_dir="./distilbert-ner-chunking",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
)

# The data collator will dynamically pad our inputs and labels to the maximum length of a batch
data_collator = DataCollatorForTokenClassification(tokenizer)

# Note: I am taking a smaller subset of the dataset here just so it runs quickly if you want to test it locally.
# In a full assignment, you would train on `tokenized_datasets['train']`.
train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(2000)) 
eval_dataset = tokenized_datasets["validation"].shuffle(seed=42).select(range(500))

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Starting training...")
# trainer.train()  # Uncomment this to run the training!

# Save the final model
# trainer.save_model("./my_chunking_model")

## 5. Evaluation

Standard accuracy is misleading because 'Outside' (`O`) tags dominate. 
Instead, we use **seqeval** to compute Precision, Recall, and F1-score for whole chunks.

In [ ]:
import evaluate
import numpy as np

metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens) and convert to string labels
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# To evaluate during training, we would have passed compute_metrics to the Trainer:
# trainer.compute_metrics = compute_metrics
# trainer.evaluate()

### Metrics Summary
- **Precision**: Accuracy of predicted chunks.
- **Recall**: Percentage of actual chunks found.
- **F1-Score**: Harmonic mean of both (our main metric).

## 6. Inference

Let's test the model on a custom sentence using the `pipeline` API.

In [ ]:
from transformers import pipeline

# We will load the base model here for demonstration, but normally you 
# would load your fine-tuned model path: pipeline('token-classification', model='./my_chunking_model')
token_classifier = pipeline("token-classification", model="distilbert-base-uncased", aggregation_strategy="simple")

sentence = "John works at Google in California."
predictions = token_classifier(sentence)

print(f"Sentence: {sentence}\n")
print("{:<15} | {:<15} | {:<10}".format("Word", "Predicted Tag", "Confidence"))
print("-" * 45)

for pred in predictions:
    # Note: Since the base distilbert-base-uncased isn't trained for chunking yet, 
    # it will output arbitrary default labels. Once you run trainer.train(), 
    # you would load that specific model and see correct 'B-NP', 'I-VP' tags here.
    print("{:<15} | {:<15} | {:.4f}".format(pred['word'], pred['entity_group'], pred['score']))

## 7. POS Tagging vs Chunking

| Feature | POS Tagging | Chunking |
| :--- | :--- | :--- |
| **Focus** | Word-level (e.g., Noun, Verb) | Phrase-level (e.g., Noun Phrase) |
| **Example** | `John(NNP) works(VBZ)` | `[John](NP) [works](VP)` |
| **Complexity**| Simpler, relies on neighbors. | Harder, uses IOB format to find boundaries. |
| **Use Cases**| Base feature engineering. | NER, Information Extraction. |

## 8. Report & Conclusion

**Observations:**
1. **Preprocessing is tricky**: Aligning subwords with `-100` was hard to grasp but crucial for transformers.
2. **DistilBERT rocks**: Much faster and lighter than BERT for local testing.
3. **Seqeval matters**: F1-score is way better than accuracy here.

Overall, getting the data aligned is 80% of the work; the Trainer API handles the rest.